# `Устанавливаем необходимые зависимости`

In [ ]:
! pip3 install -q pyspark pyarrow parquet-tools

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


## Выполнил Фирсов Федор

# `Готовим SparkContext`

Oбъект SparkContext является точкой входа для работы со Spark-кластером.

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext

In [ ]:
# Создаём конфигурационный класс с параметрами подключения
conf = (
    SparkConf()
        # Указываем URL master ноды Spark кластера
        # Можно использовать local mode, указав `local[<number_cores>]`
        # В таком случае вся обработка будет происходить на текущем компьютере
        # При этом, это может давать преимущество ввиду наличия параллелизма по ядрам компьютера
        .setMaster('local[*]')
)

# Создаём точку доступа на кластер. Позволяет использовать RDD API
sc = SparkContext(conf=conf)

# Точка доступа для использования DataFrame API
spark = SparkSession(sc)

# По завершении программы нужно обязательно выполнить остановку подключения для освобождения занятых ресурсов
# sc.stop()

# `Загрузка данных`

In [ ]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet

--2026-03-18 17:08:42--  https://github.com/evgpat/datasets/raw/refs/heads/main/calendar.parquet
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet [following]
--2026-03-18 17:08:42--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/calendar.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24369 (24K) [application/octet-stream]
Saving to: ‘calendar.parquet’

calendar.parquet    100%[===================>]  23.80K  --.-KB/s    in 0.001s  

2026-03-18 17:08:42 (17.1 MB/s) - ‘calendar.parquet’ saved [24369/24369]



Атрибуты датасета **calendar.parquet**.

Датасет содержит календарные данные, которые помогают связать продажи товаров с конкретными днями, неделями и событиями.

1. **date**: Дата записи данных.

2. **wm_yr_wk**: Календарная неделя в формате года и номера недели (год-неделя).

3. **weekday**: Название дня недели.

4. **wday**: Порядковый номер дня недели.

5. **month**: Порядковый номер месяца.

6. **year**: Год наблюдения.

7. **d**: Идентификатор дня в формате последовательности (например, d_1, d_2 и т.д.).

8. **event_name_1**: Название первичного события (праздники, акции, особые события).

9. **event_type_1**: Тип первичного события (например, праздник, спортивное событие).

10. **event_name_2**: Название вторичного события (если в этот день несколько значимых событий).

11. **event_type_2**: Тип вторичного события.

12. **snap_CA, snap_TX, snap_WI**: Индикаторы (0 или 1) программы SNAP (льготная покупка продуктов) в соответствующих штатах (Калифорния, Техас, Висконсин). Показатель 1 означает, что в указанный день были доступны льготы SNAP.

In [ ]:
# Считаем файл calendar.parquet и запишем данные в DataFrame с названием df_calendar
df_calendar = spark.read.parquet("calendar.parquet")
df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|
|2011-02-02|   11101|Wednesday|   5|    2|2011|d_5|        NULL|        NULL|        NULL|        NULL| 

In [ ]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet

--2026-03-18 17:09:00--  https://github.com/evgpat/datasets/raw/refs/heads/main/sales.parquet
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet [following]
--2026-03-18 17:09:00--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/sales.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31179351 (30M) [application/octet-stream]
Saving to: ‘sales.parquet’

sales.parquet       100%[===================>]  29.73M   172MB/s    in 0.2s    

2026-03-18 17:09:01 (172 MB/s) - ‘sales.parquet’ saved [31179351/31179351]



Атрибуты датасета **sales.parquet**.

Данный датасет содержит историю продаж товаров в розничных магазинах Walmart. Используется для анализа спроса, прогнозирования продаж и оптимизации запасов.

1. **id**: Уникальный идентификатор товара в конкретном магазине.

2. **item_id**: Идентификатор товара.

3. **dept_id**: Идентификатор отдела, к которому относится товар.

4. **cat_id**: Категория товара.

5. **store_id**: Идентификатор магазина, в котором был продан товар.

6. **state_id**: Штат, в котором расположен магазин.

7. **d_1, d_2, ..., d_n**: Ежедневные данные о продажах данного товара в указанном магазине (количество проданных единиц за каждый день). Каждый атрибут соответствует одному дню, начиная с первого дня периода наблюдения.

In [ ]:
# Считаем файл sales.parquet и запишем данные в DataFrame с названием df_sales
df_sales = spark.read.parquet("sales.parquet")
df_sales.show(5)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [ ]:
!wget https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet

--2026-03-18 17:12:18--  https://github.com/evgpat/datasets/raw/refs/heads/main/prices.parquet
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet [following]
--2026-03-18 17:12:19--  https://raw.githubusercontent.com/evgpat/datasets/refs/heads/main/prices.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2385868 (2.3M) [application/octet-stream]
Saving to: ‘prices.parquet’

prices.parquet      100%[===================>]   2.27M  --.-KB/s    in 0.07s   

2026-03-18 17:12:19 (31.7 MB/s) - ‘prices.parquet’ saved [2385868/2385868]



Атрибуты датасета **prices.parquet**.

Датасет содержит информацию о динамике цен на товары в различных магазинах за определенные недели.

1. **store_id** – идентификатор магазина, в котором продаётся товар.

2. **item_id** – уникальный идентификатор конкретного товара.

3. **wm_yr_wk** – календарная неделя года, к которой относится указанная цена (в формате Walmart-календаря).

4. **sell_price** – розничная цена продажи товара в указанном магазине в течение соответствующей недели.

In [ ]:
# Считаем файл prices.parquet и запишем данные в DataFrame с названием df_prices
df_prices = spark.read.parquet("prices.parquet")
df_prices.show(5)

+--------+-------------+--------+----------+
|store_id|      item_id|wm_yr_wk|sell_price|
+--------+-------------+--------+----------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|
|    CA_1|HOBBIES_1_001|   11326|      9.58|
|    CA_1|HOBBIES_1_001|   11327|      8.26|
|    CA_1|HOBBIES_1_001|   11328|      8.26|
|    CA_1|HOBBIES_1_001|   11329|      8.26|
+--------+-------------+--------+----------+
only showing top 5 rows


# `Задачи DataFrame API`

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [ ]:
# 1. Создайте новую колонку 'is_weekend' в df_calendar, которая показывает выходной день (Saturday or Saturday - 1, в ином случае 0).
# Используйте только DataFrame API и функции, импортированные из F.
# Подсказка: Используйте функции: when, isin.
df_calendar = df_calendar.withColumn(
    'is_weekend',
    F.when(F.col('weekday').isin(['Saturday', 'Sunday']), 1).otherwise(0)
)

df_calendar.show(5)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|   11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|   11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|   11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|   11101|Wednes

In [ ]:
# 2. Переименуйте колонку 'wm_yr_wk' в 'week_id' в df_calendar
df_calendar = df_calendar.withColumnRenamed('wm_yr_wk', 'week_id')
df_calendar.show(5)

+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|      date|week_id|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|is_weekend|
+----------+-------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+----------+
|2011-01-29|  11101| Saturday|   1|    1|2011|d_1|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-30|  11101|   Sunday|   2|    1|2011|d_2|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         1|
|2011-01-31|  11101|   Monday|   3|    1|2011|d_3|        NULL|        NULL|        NULL|        NULL|      0|      0|      0|         0|
|2011-02-01|  11101|  Tuesday|   4|    2|2011|d_4|        NULL|        NULL|        NULL|        NULL|      1|      1|      0|         0|
|2011-02-02|  11101|Wednesday|   5

In [ ]:
# 3. Выведите уникальные значения weekday в df_calendar
df_calendar.select('weekday').distinct().show()

+---------+
|  weekday|
+---------+
|Wednesday|
|  Tuesday|
|   Friday|
| Thursday|
| Saturday|
|   Monday|
|   Sunday|
+---------+



In [ ]:
# 4. Выведите количество уникальных магазинов в df_sales
# Подсказка: используйте атрибут 'store_id'
df_sales.select('store_id').distinct().count()


10

In [ ]:
# 5. Выведите среднее количество продаж по категориям товаров за d_1
# Подсказка: используйте датасет df_sales
df_sales.groupBy('cat_id').agg(F.avg('d_1').alias('avg_sales_d1')).show()

+---------+------------------+
|   cat_id|      avg_sales_d1|
+---------+------------------+
|    FOODS|1.6129436325678497|
|HOUSEHOLD|0.5433619866284622|
|  HOBBIES|0.6661946902654867|
+---------+------------------+



In [ ]:
# 6. Создайте новый атрибут 'sell_price_int' в df_prices, изменив тип данных атрибута 'sell_price' на Integer
# Подсказка, для изменения типа используйте метод cast
df_prices = df_prices.withColumn('sell_price_int', F.col('sell_price').cast('integer'))
df_prices.show(5)

+--------+-------------+--------+----------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+-------------+--------+----------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|
+--------+-------------+--------+----------+--------------+
only showing top 5 rows


In [ ]:
# 7. Выведите 5 самых дорогих товаров в df_prices
df_prices.orderBy(F.col('sell_price').desc()).show(5)

+--------+---------------+--------+----------+--------------+
|store_id|        item_id|wm_yr_wk|sell_price|sell_price_int|
+--------+---------------+--------+----------+--------------+
|    WI_3|HOUSEHOLD_2_406|   11317|    107.32|           107|
|    WI_3|HOUSEHOLD_2_406|   11318|    107.32|           107|
|    WI_3|HOUSEHOLD_2_406|   11319|    107.32|           107|
|    WI_2|HOUSEHOLD_2_406|   11242|     61.46|            61|
|    WI_2|HOUSEHOLD_2_406|   11247|     61.46|            61|
+--------+---------------+--------+----------+--------------+
only showing top 5 rows


In [ ]:
# 8. Используя sql, найдите среднее количество продаж по штатам (назвав атрибут 'avg_sales') для d1
df_sales.createOrReplaceTempView("sales")
spark.sql("SELECT state_id, AVG(d_1) AS avg_sales FROM sales GROUP BY state_id").show()

+--------+------------------+
|state_id|         avg_sales|
+--------+------------------+
|      CA|1.1639061987536898|
|      TX|1.0318137094129223|
|      WI|0.9837105061768886|
+--------+------------------+



In [ ]:
# 9. Выведите 10 строк датасета с товарами, проданных в Техасе
# Подсказка: используйте датасет df_sales
df_sales.filter(F.col('state_id') == 'TX').show(10)

+--------------------+-------------+---------+-------+--------+--------+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----

In [ ]:
# 10. Выведите 5 самых высоких цен на товары (sell_price), которые были в период с "2016-01-01" по "2016-01-31"
# Подсказка: используйте join, используйте метод between, используйте метод distinct() для уникальности цен
(
    df_prices
    .join(df_calendar, df_prices['wm_yr_wk'] == df_calendar['week_id'], 'inner')
    .filter(F.col('date').between('2016-01-01', '2016-01-31'))
    .select('sell_price')
    .distinct()
    .orderBy(F.col('sell_price').desc())
    .show(5)
)

+----------+
|sell_price|
+----------+
|     29.97|
|     29.96|
|     28.96|
|     27.98|
|     26.98|
+----------+
only showing top 5 rows


In [ ]:
# 11. Найдите товары с наибольшими продажами в каждой категории (cat_id) за d_1, используя датасет df_sales
# Подсказка: используйте оконную функцию rank()
window_spec = Window.partitionBy('cat_id').orderBy(F.col('d_1').desc())

df_sales.withColumn('rank', F.rank().over(window_spec)).filter(F.col('rank') == 1).select('cat_id', 'item_id', 'd_1', 'rank').show()

+---------+---------------+---+----+
|   cat_id|        item_id|d_1|rank|
+---------+---------------+---+----+
|    FOODS|    FOODS_3_318|360|   1|
|  HOBBIES|  HOBBIES_1_256| 54|   1|
|HOUSEHOLD|HOUSEHOLD_1_373| 44|   1|
+---------+---------------+---+----+



In [ ]:
# 12. Создайте новую колонку "price_category" в df_prices, указав высокая или низкая цена: если цена больше 5, то "High", иначе "Low"
# Подсказка: используйте конструкцию when().otherwise
df_prices = df_prices.withColumn(
    'price_category',
    F.when(F.col('sell_price') > 5, 'High').otherwise('Low')
)
df_prices.show(20)

+--------+-------------+--------+----------+--------------+--------------+
|store_id|      item_id|wm_yr_wk|sell_price|sell_price_int|price_category|
+--------+-------------+--------+----------+--------------+--------------+
|    CA_1|HOBBIES_1_001|   11325|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11326|      9.58|             9|          High|
|    CA_1|HOBBIES_1_001|   11327|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11328|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11329|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11330|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11331|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11332|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11333|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001|   11334|      8.26|             8|          High|
|    CA_1|HOBBIES_1_001| 

In [ ]:
# 13. Вычислите суммарные продажи по всем магазинам (store_id) за d_1
df_sales.groupBy('store_id').agg(F.sum('d_1').alias('total_sales_d1')).show()


+--------+--------------+
|store_id|total_sales_d1|
+--------+--------------+
|    TX_2|          3852|
|    TX_1|          2556|
|    CA_4|          1625|
|    CA_2|          3494|
|    CA_1|          4337|
|    CA_3|          4739|
|    WI_2|          2256|
|    WI_3|          4038|
|    WI_1|          2704|
|    TX_3|          3030|
+--------+--------------+



# `Задачи RDD`

### Данные

Файл - `transaction.csv`

Формат записей:

```
user_id, timestamp, item_id, category, price, quantity, city
```





In [ ]:
# Задание 0. Загрузите данные в RDD
rdd = sc.textFile("/content/transactions.csv")
header = rdd.first()
rdd = rdd.filter(lambda line: line != header).map(lambda line: line.split(","))
rdd.take(1)

[['22', '2025-03-07 15:41', 'E22', 'beauty', '1227', '1', 'Perm']]

In [ ]:

# Задание 1. Найти ТОП-5 категорий по общей выручке

category_revenue = rdd.map(lambda x: (x[3].strip(), float(x[4].strip()) * float(x[5].strip()))).reduceByKey(lambda a, b: a + b)

top5 = category_revenue.sortBy(lambda x: x[1], ascending=False).take(5)

for x in top5:
    print(x)

('beauty', 453070.0)
('home', 444245.0)
('kids', 403922.0)
('sport', 389754.0)
('toys', 381717.0)


In [ ]:


# Задание 2. Найти пользователей, покупавших в более чем двух категориях

user_cat = rdd.map(lambda x: (x[0].strip(), x[3].strip())).distinct().groupByKey().mapValues(set)

users_many = user_cat.filter(lambda x: len(x[1]) > 2)

for u in users_many.take(10):
    print(u[0], " → ", len(u[1]), "категорий:", u[1])




22  →  6 категорий: {'auto', 'garden', 'beauty', 'kids', 'toys', 'electronics'}
45  →  6 категорий: {'books', 'auto', 'kids', 'sport', 'toys', 'electronics'}
62  →  7 категорий: {'home', 'garden', 'beauty', 'kids', 'sport', 'toys', 'electronics'}
79  →  8 категорий: {'books', 'home', 'auto', 'garden', 'beauty', 'kids', 'sport', 'electronics'}
18  →  8 категорий: {'books', 'home', 'auto', 'garden', 'kids', 'sport', 'toys', 'electronics'}
64  →  9 категорий: {'books', 'home', 'auto', 'toys', 'garden', 'beauty', 'kids', 'sport', 'electronics'}
44  →  5 категорий: {'home', 'toys', 'kids', 'sport', 'electronics'}
13  →  8 категорий: {'books', 'home', 'garden', 'beauty', 'kids', 'sport', 'toys', 'electronics'}
21  →  9 категорий: {'books', 'home', 'auto', 'garden', 'beauty', 'kids', 'sport', 'toys', 'electronics'}
15  →  7 категорий: {'books', 'home', 'toys', 'garden', 'kids', 'sport', 'electronics'}


In [ ]:
# Задание 3. Найти средний чек по каждому городу

city_pairs = rdd.map(lambda x: (x[6].strip(), (float(x[4].strip()) * float(x[5].strip()), 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

sorted_avg = city_pairs.mapValues(lambda x: (x[0] / x[1], 2)).sortBy(lambda x: x[1], ascending=False)

for x in sorted_avg.collect():
    print(x)




('Ekaterinburg', (2639.1736526946106, 2))
('Ufa', (2435.830188679245, 2))
('Kazan', (2366.483660130719, 2))
('Rostov', (2353.311111111111, 2))
('Moscow', (2326.259493670886, 2))
('Novosibirsk', (2292.322033898305, 2))
('Samara', (2219.048780487805, 2))
('Perm', (2216.308988764045, 2))
('SPB', (2148.1158536585367, 2))


In [ ]:
# Задание 4. Найти самый продаваемый товар внутри каждой категории и
# суммарное количество проданных единиц этого товара

cat_item = rdd.map(lambda x: ((x[3].strip(), x[2].strip()), int(x[5].strip()))).reduceByKey(lambda a, b: a + b)

top_items = cat_item.map(lambda x: (x[0][0], (x[0][1], x[1]))).reduceByKey(lambda a, b: a if a[1] >= b[1] else b)

for x in top_items.collect():
    print(x)


('toys', ('D14', 493))
('beauty', ('E22', 568))
('auto', ('Q77', 452))
('kids', ('H55', 517))
('garden', ('G10', 448))
('sport', ('C02', 476))
('home', ('F99', 590))
('books', ('B07', 465))
('electronics', ('A13', 475))


In [ ]:
# Задание 5. Определите час суток с максимальной выручкой.
# Найдите выручку за этот час и определите ее долю в общей выручке.
# Подсказка: создайте и используйте функцию для парсинга часа из поля timestamp

from datetime import datetime

def extract_hour(ts):
    return int(ts.strip()[11:13])

hour_rev = rdd.map(lambda x: (extract_hour(x[1]), float(x[4].strip()) * float(x[5].strip()))) \
              .reduceByKey(lambda a, b: a + b)

peak_hour, peak_revenue = hour_rev.sortBy(lambda x: x[1], ascending=False).first()
total_revenue = hour_rev.map(lambda x: x[1]).sum()
peak_share = peak_revenue / total_revenue

print("Час пик:", (peak_hour, peak_revenue))
print("Доля общего оборота:", peak_share)



Час пик: (11, 181001.0)
Доля общего оборота: 0.05174835571222927
